# Import Module

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Mempersiapkan Data

In [14]:
# Daca data csv
df = pd.read_csv('/content/drive/MyDrive/Data Fluks/Hasil/Data_C-Flux_new_parameter')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.head()

,timestamp,Air Pressure (kPa),Air Temperature (Celcius),Altitude (MSAL),Relatif Humidity (%),Temperature (Celsius),C02 mole fraction (µmol/mol_a),CH4 mole fraction (µmol/mol_a),Water Vapor Mass Density (kg/m^3),utara (m/s),...,Molar Volume of Air (m^3/mol),Saturation Vapor Pressure (kPa),Saturation Vapor Pressure (Pa),Water Vapor Partial Pressure (kPa),Water Vapor Partial Pressure (Pa),Dew point temperature (K),H2O Mole Fraction (mmol/mol_a),H2O Mixing Ratio (mmol/mol_d),CO2 Mixing Ratio (µmol/mol_d),w (m/s)
0,2024-06-10 13:06:01.353,100.829346,23.213638,37.528976,74.532692,27.232393,364.0,0,1943.793818,2.79,...,0.024770,3.612635,3612.635210,2.692594,2692.594272,295.471102,26.704470,27.437165,364.974645,-4.60
1,2024-06-10 13:06:01.436,100.875145,27.721346,37.549667,74.512856,27.275118,363.0,0,1947.872620,2.79,...,0.024762,3.621695,3621.694671,2.698628,2698.628118,295.507897,26.752161,27.487511,363.973708,-4.86
2,2024-06-10 13:06:01.600,100.876099,27.732904,37.537556,74.476234,27.275118,363.0,0,1946.915292,2.79,...,0.024762,3.621695,3621.694671,2.697302,2697.301814,295.499815,26.738760,27.473364,363.973219,-4.32
3,2024-06-10 13:06:01.763,100.874657,27.744940,37.568340,74.431984,27.259096,363.0,0,1944.035748,2.79,...,0.024761,3.618295,3618.295054,2.693169,2693.168794,295.474609,26.698171,27.430515,363.971738,-4.86
4,2024-06-10 13:06:01.926,100.874779,27.753908,37.621830,74.491493,27.301826,363.0,0,1950.191300,2.79,...,0.024764,3.627368,3627.368116,2.702081,2702.080674,295.528919,26.786484,27.523749,363.974961,-4.86


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1945487 entries, 0 to 1945486
Data columns (total 29 columns):
 #   Column                              Dtype         
---  ------                              -----         
 0   timestamp                           datetime64[ns]
 1   Air Pressure (kPa)                  float64       
 2   Air Temperature (Celcius)           float64       
 3   Altitude (MSAL)                     float64       
 4   Relatif Humidity (%)                float64       
 5   Temperature (Celsius)               float64       
 6   C02 mole fraction (µmol/mol_a)      float64       
 7   CH4 mole fraction (µmol/mol_a)      int64         
 8   Water Vapor Mass Density (kg/m^3)   float64       
 9   utara (m/s)                         float64       
 10  barat (m/s)                         float64       
 11  timur (m/s)                         float64       
 12  selatan (m/s)                       float64       
 13  bawah (m/s)                         float6

In [16]:
# Mengurutkan kembali timestamp
df = df.sort_values("timestamp").reset_index(drop=True)

In [17]:
# 2️⃣ Tentukan batch 30 menit
batch_duration = '30T'

# Gunakan pd.Grouper untuk resample per 30 menit
df_30min = df.groupby(pd.Grouper(key='timestamp', freq=batch_duration))

/tmp/ipython-input-2048626054.py:5: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_30min = df.groupby(pd.Grouper(key='timestamp', freq=batch_duration))


In [18]:
# 3️⃣ Hitung jumlah data dan persentase
sensing_rate_hz = 7
seconds_per_30min = 30 * 60
ideal_count = sensing_rate_hz * seconds_per_30min

batch_summary = df_30min.agg({
    'timestamp': 'count',
    **{col: 'mean' for col in df.select_dtypes(include=['float64', 'int64']).columns}
}).rename(columns={'timestamp': 'actual_count'}).reset_index()

batch_summary['ideal_count'] = ideal_count
batch_summary['missing_data'] = batch_summary['ideal_count'] - batch_summary['actual_count']
batch_summary['accuracy_percent'] = (batch_summary['actual_count'] / batch_summary['ideal_count'] * 100).round(2)

In [20]:
# 4️⃣ Covarian dan fluks karbon
cov_list = []
for b_start, group in df_30min:
    if len(group) > 0:
        group['w_prime'] = group['w (m/s)'] - group['w (m/s)'].mean()
        group['CO2_prime'] = group['C02 mole fraction (µmol/mol_a)'] - group['C02 mole fraction (µmol/mol_a)'].mean()
        cov_list.append(np.mean(group['w_prime'] * group['CO2_prime']))
    else:
        cov_list.append(np.nan)
batch_summary['cov_w_CO2'] = cov_list

/tmp/ipython-input-3915014857.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group['w_prime'] = group['w (m/s)'] - group['w (m/s)'].mean()
/tmp/ipython-input-3915014857.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group['CO2_prime'] = group['C02 mole fraction (µmol/mol_a)'] - group['C02 mole fraction (µmol/mol_a)'].mean()
/tmp/ipython-input-3915014857.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value 

In [21]:
# 5️⃣ Fluks karbon
batch_summary['carbon_flux'] = batch_summary['cov_w_CO2'] * batch_summary['Molar Density of Air (mol/m^3)']

# 6️⃣ Tampilkan beberapa baris
batch_summary.head()

,timestamp,actual_count,Air Pressure (kPa),Air Temperature (Celcius),Altitude (MSAL),Relatif Humidity (%),Temperature (Celsius),C02 mole fraction (µmol/mol_a),CH4 mole fraction (µmol/mol_a),Water Vapor Mass Density (kg/m^3),...,Dew point temperature (K),H2O Mole Fraction (mmol/mol_a),H2O Mixing Ratio (mmol/mol_d),CO2 Mixing Ratio (µmol/mol_d),w (m/s),ideal_count,missing_data,accuracy_percent,cov_w_CO2,carbon_flux
0,2024-06-10 13:00:00,8832,100.827531,26.376570,41.500479,81.746555,26.658762,360.036798,0.0,2064.145237,...,296.423658,28.300732,29.125569,361.058579,-5.888737,12600,3768,70.10,-1.298012,-52.502474
1,2024-06-10 13:30:00,2218,100.775620,27.077859,45.841672,84.466928,27.118138,354.093327,0.0,2189.055824,...,297.430246,30.078594,31.011668,355.161609,-4.973237,12600,10382,17.60,-0.015544,-0.627457
2,2024-06-10 14:00:00,1,100.800875,26.889366,45.717457,81.821922,28.754484,351.000000,0.0,2320.151137,...,298.497538,32.045554,33.106469,352.128415,-3.915409,12600,12599,0.01,0.000000,0.000000
3,2024-06-10 14:30:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,12600,12600,0.00,NaN,NaN
4,2024-06-10 15:00:00,1469,100.824257,26.169551,41.777170,91.417568,26.450613,351.942138,0.0,2282.975962,...,298.097490,31.284210,32.294550,353.046623,-3.100497,12600,11131,11.66,0.041789,1.691405


In [22]:
batch_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 358 entries, 0 to 357
Data columns (total 35 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   timestamp                           358 non-null    datetime64[ns]
 1   actual_count                        358 non-null    int64         
 2   Air Pressure (kPa)                  356 non-null    float64       
 3   Air Temperature (Celcius)           356 non-null    float64       
 4   Altitude (MSAL)                     356 non-null    float64       
 5   Relatif Humidity (%)                356 non-null    float64       
 6   Temperature (Celsius)               356 non-null    float64       
 7   C02 mole fraction (µmol/mol_a)      356 non-null    float64       
 8   CH4 mole fraction (µmol/mol_a)      356 non-null    float64       
 9   Water Vapor Mass Density (kg/m^3)   356 non-null    float64       
 10  utara (m/s)               